# Option E — Jupyter widget sim-vs-real scrubber

ipywidgets slider over the matplotlib panels. The slider drags a vertical cursor across the time-series traces and a pose marker along the integrated trajectory. The 'before' picture in the friction-to-Quix-antidote narrative.

Segment: `063c5f30b8e68fae / 00000000--cf682901f4 / 1` (2900 rows @ 50 Hz, ~58 s).

Run `Cell > Run All`, then drag the slider at the bottom.

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline

KB003 = Path.cwd().resolve()
while KB003.name != "KB003":
    KB003 = KB003.parent

SEGMENT_CSV = KB003 / "simdata/segments/TESLA_MODEL_3/063c5f30b8e68fae/00000000--cf682901f4/1/sim.csv"
df = pd.read_csv(SEGMENT_CSV)
t = df["t_s"].to_numpy()
print(f"Loaded {len(df)} rows, {t[-1]:.1f} s @ 50 Hz")
df.head()

In [ ]:
REAL = "#d62728"
SIM = "#1f77b4"

def render(idx: int):
    fig, axes = plt.subplots(3, 2, figsize=(13, 9))
    fig.suptitle(f"t = {t[idx]:.2f} s   (sample {idx}/{len(df)-1})", fontsize=12)

    # 1. Trajectory with moving pose marker
    ax = axes[0, 0]
    ax.plot(df["x_m"], df["y_m"], color=SIM, lw=1.2)
    psi = df["psi_rad"].iat[idx]
    x, y = df["x_m"].iat[idx], df["y_m"].iat[idx]
    ax.scatter([x], [y], c="orange", s=80, zorder=5, edgecolor="black")
    # heading arrow
    ax.arrow(x, y, 4*math.cos(psi), 4*math.sin(psi),
             head_width=2.0, head_length=2.0, fc="orange", ec="black", zorder=6)
    ax.set_aspect("equal", adjustable="datalim"); ax.grid(True, alpha=0.3)
    ax.set_title("Trajectory + pose"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")

    def trace(ax, title, ylabel, series):
        for label, vals, color, style in series:
            ax.plot(t, vals, color=color, lw=1.1, ls=style, label=label)
        ax.axvline(t[idx], color="black", lw=0.8, alpha=0.6)
        ax.set_title(title); ax.set_xlabel("t [s]"); ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3); ax.legend(loc="best", fontsize=8)

    trace(axes[0, 1], "Speed — real vs sim", "m/s", [
        ("real", df["v_mps"], REAL, "-"),
        ("sim",  df["v_state_mps"], SIM, "--"),
    ])
    trace(axes[1, 0], "Road-wheel steering — real vs sim", "rad", [
        ("real", df["delta_road_rad"], REAL, "-"),
        ("sim",  df["delta_state_rad"], SIM, "--"),
    ])
    trace(axes[1, 1], "Longitudinal accel (input)", "m/s²", [
        ("real", df["a_long_mps2"], REAL, "-"),
    ])
    trace(axes[2, 0], "Yaw rate — sim only", "rad/s", [
        ("sim", df["psi_dot_rads"], SIM, "-"),
    ])
    trace(axes[2, 1], "Lateral accel — sim only", "m/s²", [
        ("sim", df["a_y_mps2"], SIM, "-"),
    ])

    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

slider = widgets.IntSlider(value=0, min=0, max=len(df)-1, step=10,
                           description="sample", continuous_update=False,
                           layout=widgets.Layout(width="90%"))
out = widgets.interactive_output(render, {"idx": slider})
display(slider, out)